![](https://carleton.ca/ses/wp-content/uploads/Students-taking-exam2.jpg)

<font color="lightseagreen" size=+3.5><b> Student's Test Performance: What Factors Affect Test Scores?</b></font>

---

<a id="toc"></a>


<font color="lightseagreen" size=+2.5><b>Table of Contents</b></font>

    
* [1. Introduction](#1)

<font color="lightgray" size=+1.0><b>Part-1: EDA</b></font>


* [2. Data Overview](#2)
* [3. Features distrubution](#3)
    - [3.1 How is gender distributed in other categories?](#3.1)
* [4. Test scores distribution](#4)
* [5. Initial observations](#5)
    - [5.1 Who benefited from test preparation course the most?](#5.1)
    - [5.2 What is the effect of lunch type on test results?](#5.2)
* [6. Test score correlation](#6)
* [7. Correlation heatmap](#7)
* [8. Conclusions: EDA](#8)


<font color="lightgray" size=+1.0><b>Part-2: Model</b></font>


* [9. Modeling and prediction](#9)
* [10. Conclusions: Model](#10)
* [11. Closing remarks](#11)
* [12. References](#12)

---   


<a id="1"></a>
<font color="lightseagreen" size=+2.5><b>1. Introduction</b></font>

In this notebook factors that influence the outcome of a test score of students at a public school is explored. The dataset used for this study has a variety of personal, social, and economic factors that have interaction effects upon them. The test score includes three test subjects, which are `math`, `reading` and `writing`. Independent features such as gender, race/ethnic group, parents educational background, lunch type and test preparation course are explored. 

We can further divide the independent parameters into two groups - **inherent attributes** (those we do not have control over) such as gender, race or ethnicity and parental education level; and **acquired attributes** (those we can have control over) such as the test preparation course and lunch type. Thus by studying which features have a significant effect on test score, parents/care-takers or policy makers know where to focus their efforts and logistics on. 

<font color="lightseagreen" size=+1.50><b>Questions we would like to answer using this dataset include:</b></font>

- How effective is the test preparation course?
- Which major factors contribute to test outcomes?
- What would be the best way to improve student scores on each test?

<font color="lightseagreen" size=+1.50><b>Remarks:</b></font>
- This dataset is `FICTIONAL`! Does not represent real school data. **The creator of the dataset generator function** (see the ref) made it for educational purposes. So be informed! 
- The author of the dataset (kaggle dataset)  has only published one download (1000) from the source data generator function. I downloaded 9 more rounds and this notebook has 9000 more data (10k in total).
---

<font color="lightgray" size=+3.0><b>Part-1: EDA</b></font>

<a id="2"></a>

<font color="lightseagreen" size=+2.5><b>2. Data Overview</b></font>

<font color="lightseagreen" size=+1.0><b>Import</b></font>

In [ ]:
import numpy as np 
from scipy import stats
import pandas as pd
import seaborn as sns
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
import plotly.express as pex
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True)

import warnings
warnings.filterwarnings('ignore')

<font color="lightseagreen" size=+1.0><b>Download and combine datasets</b></font>

- We have two datasets (`students-performance-in-exams` and `more-exam-data`). Essentially they are the same datasets, the only difference is that the `more-exam-data` has more data, i.e 10 X more. So we combine these to datasets in the below section. Note that I created the `more-exam-data` set using the same generator as the other dataset. 

<font color="lightseagreen" size=+1.0><b>Dataset overview:</b></font>

- There are 10 000 rows and 8 columns. 
- Three of the columns are `integer` data-type whereas five of the remaining are of `object` type.


<font color="lightseagreen" size=+1.0><b>Column Description:</b></font>

- `gender`: the gender of the student (male/female)
- `race/ethnicity`: the ethnic group or race the student belong to (group A to E)
- `parental level of education`: the highest education level attained by the parents of the student (from some high school to master's degree)
- `lunch`: the type of lunch package the student has (standard or free/reduced)
- `test preparation course`: if the student has a test preparation course or not (yes/no)
- `math score`: the student's score on math test (int 0 to 100)
- `reading score`: the student's score on reading test (int 0 to 100)
- `writing score`: the student's score on writing test (int 0 to 100)

In [ ]:
data = pd.read_csv(f'/kaggle/input/students-performance-in-exams/StudentsPerformance.csv')

df_extra = []
for i in range(9):
    df = pd.read_csv(f'/kaggle/input/more-exam-data/exams ({i+1}).csv')
    df_extra.append(df)
    
data_total = pd.concat([data, pd.concat(df_extra)], axis=0)

data = data_total.copy()
display(data.info())
 
display(data.head())


<font color="lightseagreen" size=+1.0><b>Check for null values</b></font>

- There are no null values in this dataset. So no missing values imputation is required.

In [ ]:
print('There are a total {} missing values in the dataset.'.format(data.isnull().any().sum()))

<font color="lightseagreen" size=+1.0><b>Check for duplicates</b></font>

As outlined in the introduction section, the dataset is synthetically generated. Moreover, while creating the second dataset (`more-exam-data`), I have requested the data generation engine 10X. Thus it is likely that our combined dataset will have some duplicates. Let's check if there are any. 
- There are 46 duplicate rows in the data. 
- We will drop them before we proceed with our explorations.


In [ ]:
print('There are {} duplicated rows in the dataset.'.format(data.duplicated().sum()))
data.drop_duplicates(inplace=True)
print('Number of rows after dropping duplicates is {}.'.format(data.shape[0]))

<font color="lightseagreen" size=+1.0><b>Additional Feature</b></font>

- Here we will add the average of the test subjects as an additional feature (`average_score`). This feature can be used for reference in the analysis that will follow in the later section of the notebook.


In [ ]:
data['average_score'] = (data['math score'] + data['reading score'] + data['writing score'])/3
data.describe()

<a id="3"></a>
<font color="lightseagreen" size=+2.5><b>3. Features distrubution</b></font>

In this section we will look at the distribution of the categorical features. Are catergories withing a feature fairly balanced? 

- `Gender`: Gender is fairly balanced - with 51% feamles to 49% males. 
- `Race/ethnicity`: Here `group C` is the most represented with 31.8%, `group A` is the least with 8.2%.
- `Parental education level`: Master's degree holder parents are the fewest represented followed by parents with bachelor's degree. The rest are in the same ball-park
- `Lunch type`: 65% of the students have standard lunch the rest is free/reduced lunch.
- `Test preparation course`: 65.6% of the students haven't had test preparation course

<a href="#toc">Back to top</a>

In [ ]:
#colors = ['lightgray', 'Rebeccapurple','gold','royalblue','lightseagreen','lightsalmon']
colors = ['gold', 'mediumturquoise', 'darkorange', 'lightgreen', 'black', 'Gray']

fig = make_subplots(rows=3, cols=2,
                    specs=[[{'type':'domain'}, {'type':'domain'}],
                           [{'type':'domain'}, {'type':'domain'}], 
                           [{'type':'domain'}, {'type':'domain'}]])


fig.add_trace(
    go.Pie(
        labels=data['gender'],
        values=None,#scalegroup='one',
        hole=.4,
        title='Gender',
        titlefont={'color':'black', 'size': 24},
        ),
    row=1,col=1
    )
fig.update_traces(
    hoverinfo='label+value',
    textinfo='label+percent',
    textfont_size=12,
    marker=dict(
        colors=colors, #['lightseagreen', 'lightsalmon'], 
        line=dict(color='#000000',
                  width=2)
        )
    )

fig.add_trace(
    go.Pie(
        labels=data['race/ethnicity'],
        values=None,#scalegroup='one',
        hole=.4,
        title='Race',
        titlefont={'color':'black', 'size': 24},
        ),
    row=1,col=2
    )
fig.update_traces(
    hoverinfo='label+value',
    textinfo='label+percent',
    textfont_size=12,
    marker=dict(
        colors=colors,#[0:6],
        line=dict(color='#000000',
                  width=2)
        )
    )

fig.add_trace(
    go.Pie(
        labels=data['parental level of education'],
        values=None,#scalegroup='one',
        hole=.4,
        title='ParentEduc.',
        titlefont={'color':'black', 'size': 24},
        ),
    row=2,col=1
    )
fig.update_traces(
    hoverinfo='label+value',
    textinfo='label+percent',
    textfont_size=12,
    marker=dict(
        colors=colors,
        line=dict(color='#000000',
                  width=2)
        )
    )

fig.add_trace(
    go.Pie(
        labels=data['lunch'],
        values=None,#scalegroup='one',
        hole=.4,
        title='Lunch',
        titlefont={'color':'black', 'size': 24},
        ),
    row=2,col=2
    )
fig.update_traces(
    hoverinfo='label+value',
    textinfo='label+percent',
    textfont_size=12,
    marker=dict(
        colors=colors, #['lightseagreen', 'lightsalmon'],
        line=dict(color='#000000',
                  width=2)
        )
    )

fig.add_trace(
    go.Pie(
        labels=data['test preparation course'],
        values=None,#scalegroup='one',
        hole=.4,
        title='TestPrep.',
        titlefont={'color':'black', 'size': 24},
       ),
    row=3,col=1
    )
fig.update_traces(
    hoverinfo='label+value',
    textinfo='label+percent',
    textfont_size=12,
    marker=dict(
        colors=colors,#['lightseagreen', 'lightsalmon'],
        line=dict(color='#000000',
                  width=2)
        )
    )
fig.layout.update(title="Independent Features Distribution", showlegend=False, height=850, width=750, 
                  template=None, titlefont={'color':'black', 'size': 24}
                 )
fig.show()


<a id="3.1"></a>
<font color="lightseagreen" size=+1.5><b>3.1. How is gender distributed in other categories?</b></font>

- The gender distribution in the individual categories of the other features is fairly even. The difference is within 1% for all categorical features except race group B and D which has upto 2% variation.

In [ ]:
##### Parental education ####
df = data
fig = px.histogram(df, 
                   x="parental level of education", 
                   y=None, color="gender", width=600, height=400,
                   barmode= 'group',
                   histnorm='percent',
                   category_orders={
                       "parental level of education": ["some high school", "high school", "associate's degree", "some college", "bachelor's degree", "master's degree"],
                       "gender": ["male", "female"]
                },
                
                color_discrete_map={ 
                    "male": "lightblue", "female": "lightsalmon"
                },
                template="simple_white"
                )

fig.update_layout(title="<b> Parental Level of Education", 
                  font_family="San Serif",
                  titlefont={'size': 24},
                  legend=dict(
                  orientation="v", y=1, yanchor="top", x=1.0, xanchor="right" )                 
                 )
fig.show()

##### race #####

fig = px.histogram(df, x="race/ethnicity", y=None, color="gender",
                width=600, height=400, barmode='group',
                histnorm='percent',
                category_orders={ 
                "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"], 
                "gender": ["male", "female"]
                },
                color_discrete_map={ 
                    "male": "lightblue", "female": "lightsalmon"
                },
                template="simple_white"
                )

fig.update_layout(title="<b> Race/Ethnic Group", 
                  font_family="San Serif",
                  titlefont={'size': 24},
                  legend=dict(
                  orientation="v", y=1, yanchor="top", x=1.0, xanchor="right" )                 
                 )

fig.show()

##### lunch #####

fig = px.histogram(df, x="lunch", y=None, color="gender",
                width=600, height=400, barmode='group',
                histnorm='percent',
                color_discrete_map={ 
                    "male": "lightblue", "female": "lightsalmon"
                },
                template="simple_white"
                )

fig.update_layout(title="<b> Lunch Type",
                  font_family="San Serif",
                  titlefont={'size': 24},
                  legend=dict(
                  orientation="v", y=1, yanchor="top", x=1.0, xanchor="right" )                 
                 )
fig.show()


#### test prep ####
fig = px.histogram(df, x="test preparation course", y=None, color="gender",
                width=600, height=400, barmode='group',
                histnorm='percent',
                color_discrete_map={ 
                    "male": "lightblue", "female": "lightsalmon"
                },
                template="simple_white"
                )

fig.update_layout(title="<b> Test Preparation Course",
                  font_family="San Serif",
                  titlefont={'size': 24},
                  legend=dict(
                  orientation="v", y=1, yanchor="top", x=1.0, xanchor="right" )                 
                 )
fig.show()



<a id="4"></a>
<font color="lightseagreen" size=+2.5><b>4. Test scores distribution</b></font>

In [ ]:
data_m = data['math score']
data_r = data['reading score']
data_w = data['writing score']

fig = go.Figure()

fig.add_trace(go.Violin(x=data_m, line_color='salmon', name='Math'))
fig.add_trace(go.Violin(x=data_r, line_color='gold', name= 'Reading'))
fig.add_trace(go.Violin(x=data_w, line_color='lightseagreen', name='Writing'))

fig.update_traces(orientation='h', side='positive', width=3, points=False, meanline_visible=True)
fig.update_layout(xaxis_showgrid=False, xaxis_zeroline=False)

fig.update_layout(title='<b> Test Score Distribution <b>',
                  titlefont={'size': 24,
                             'family':'San Serif',
                             'color': 'black'
                            },
                  xaxis_title='Test scores',
                  width=750,
                  showlegend=False,
                  paper_bgcolor="lightgray",
                  plot_bgcolor='lightgray',             
 )
fig.show()

In [ ]:
df_genderAvg = df.groupby(["gender"])['math score', 'reading score', 'writing score'].mean()
df_genderAvg

<a id="5"></a>
<font color="lightseagreen" size=+2.5><b>5. Initial observations</b></font>

- Female students performed better than male students.
- Free vs standard lunch: students who have the standard lunch packet performed better.
- Test preparation courses helped increase test scores.
- Students of race group E have the best overall performance. However, those from race group A were the lower end of the score spectrum.
- Students whose parents have master's degrees performed best. However, students whose parents have 'some high school' education level scored the lowest average. Interestingly though, for group A students having parents with master's degrees didn't help that much.**

<a href="#toc">Back to top</a>

In [ ]:
df = data
total_average = data['average_score'].mean()
fig = px.box(df, y="average_score", x="race/ethnicity",color="gender",
             title='Average score vs Race',
             template='plotly_dark', width=650,height=450,
             category_orders={ 
            "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]},
#              color_discrete_map={ 
#             "male": "RebeccaPurple", "female": "lightsalmon"}             
            )

fig.add_shape( 
    type="line", line_color="yellow", line_width=3, opacity=1, line_dash="dot",
    x0=0, x1=1, xref="paper", y0=total_average, y1=total_average, yref="y"
)
fig.update_layout(
    paper_bgcolor="#232624",
    font=dict(
        color='white'))

fig.show()


##########

fig = px.box(df, y="average_score", x="lunch",color="gender",
             title='Average score vs Lunch',
             template='plotly_dark', width=650,height=450,
             category_orders={ 
            "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]
             }           
            )

fig.add_shape( 
    type="line", line_color="yellow", line_width=3, opacity=1, line_dash="dot",
    x0=0, x1=1, xref="paper", y0=total_average, y1=total_average, yref="y"
)
fig.update_layout(
    paper_bgcolor="#232624",
    font=dict(
        color='white'))

fig.show()

###########

fig = px.box(df, y="average_score", x="test preparation course",color="gender",
             title='Average score vs TestPrep.',
             template='plotly_dark', width=650,height=450,
             category_orders={ 
            "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]
             }           
            )

fig.add_shape( 
    type="line", line_color="yellow", line_width=3, opacity=1, line_dash="dot",
    x0=0, x1=1, xref="paper", y0=total_average, y1=total_average, yref="y"
)
fig.update_layout(
    paper_bgcolor="#232624",
    font=dict(
        color='white'))

fig.show()
#######################

fig = px.box(df, y="average_score", x="parental level of education", color="gender",
             title='Average score vs ParentEduc.',
             template='plotly_dark', width=650,height=450,
             category_orders={ 
            "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]
             }           
            )

fig.add_shape( 
    type="line", line_color="yellow", line_width=3, opacity=1, line_dash="dot",
    x0=0, x1=1, xref="paper", y0=total_average, y1=total_average, yref="y"
)
fig.update_layout(
    paper_bgcolor="#232624",
    font=dict(
        color='white'))

fig.show()

########################

fig = px.box(df, y="average_score", x="parental level of education", color="race/ethnicity",
             
             title='Averge score vs ParentEduc. vs Race ',
             template='plotly_dark',
             
             width=850,height=450,
             
             category_orders={ 
             "parental level of education": ["some high school", "high school", "associate's degree",
                                            "some college", "bachelor's degree", "master's degree"],
             
             "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]             
             }
             )

fig.add_shape( 
    type="line", line_color="yellow", line_width=3, opacity=1, line_dash="dot",
    x0=0, x1=1, xref="paper", y0=total_average, y1=total_average, yref="y"
)

fig.update_layout(
    paper_bgcolor="#202624",
    font=dict(
        color ='white', 
    )
 )


fig.show()

<a id="5.1"></a>
<font color="lightseagreen" size=+1.0><b>5.1 Who benefited from test preparation course the most?</b></font>

- **Male students** benefited from test preparation course more than female students did
- **Writing test score** is where the highest points (10 points for male students) gain achieved
- Male students from **Group B and C** showed the highest points gain (writing score)
- From those race group B & C male students whose parents had '**some college**' education got the biggest advantage from the test preparation course. Male students had a 13 point gain whereas female students gained 11 points.


In [ ]:
# maths 
fig = px.box(df, 
                 x="test preparation course", y="math score", 
                 facet_col="gender",
                 color='gender',
                 template='simple_white',
                 width=750, height=350,
                 category_orders={
                     "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]
                     }                
                )

fig.update_layout(
    title="Test preparation course on math score",
    margin=dict(l=20, r=20, t=70, b=20, pad=1),
    paper_bgcolor="lightgray",
)
fig.show()

# reading 
fig = px.box(df, 
                 x="test preparation course", y="reading score", 
                 facet_col="gender",
                 color='gender',
                 template='simple_white',
                 width=750, height=350,
                 category_orders={
                     "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]
                     }                
                )

fig.update_layout(
    title="Test preparation course on reading score",
    margin=dict(l=20, r=20, t=70, b=20, pad=1),
    paper_bgcolor="lightgray",
)
fig.show()

# writing
fig = px.box(df, 
                 x="test preparation course", y="writing score", 
                 facet_col="gender",
                 color='gender',
                 template='simple_white',
                 width=750, height=350,
                 category_orders={
                     "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]
                     }                
                )

fig.update_layout(
    title="Test preparation course on writing score",
    margin=dict(l=20, r=20, t=70, b=20, pad=1),
    paper_bgcolor="lightgray",
)
fig.show()

# writing, race
fig = px.box(df, 
                 x="test preparation course", y="writing score", 
                 facet_col="race/ethnicity",
                 color='gender',
                 template='simple_white',
                 width=1500, height=350,
                 category_orders={
                     "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]
                     }                
                )

fig.update_layout(
    title="Test preparation course on writing score: variation accros race group",
    margin=dict(l=20, r=20, t=70, b=20, pad=1),
    paper_bgcolor="lightgray",
)
fig.show()


# writing, race, parent education
DF = df[(df['race/ethnicity'] == 'group B') | (df['race/ethnicity'] == 'group C')]

fig = px.box(DF, 
                 x="test preparation course", y="writing score", 
                 facet_col="parental level of education",
                 color='gender',
                 template='simple_white',
                 width=2000, height=350,
                 category_orders={
                     "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]
                     }                
                )

fig.update_layout(
    title="Test preparation course on writing score: variation across ParentEduc (group B&C)",
    margin=dict(l=20, r=20, t=70, b=20, pad=1),
    paper_bgcolor="lightgray",
)
fig.show()

<a id="5.2"></a>
<font color="lightseagreen" size=+1.0><b>5.2 What is the effect of lunch type on test results?</b></font>

- **Maths score** on both male and female students were the most affected by a lunch type. Both gender groups lost 12 points by having to resort to the free or reduced lunch.
- **Female** students of race **group A** were the most affected. Their math score suffered a whopping 15 points drop by not having the standard lunch. 
- **Female** students of race **group C** were the second most affected with a drop of 13points.


In [ ]:
# maths 
fig = px.box(df, 
                 x="lunch", y="math score", 
                 facet_col="gender",
                 color='gender',
                 template='simple_white',
                 width=900, height=400,
                 category_orders={
                     "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]
                     }                
                )

fig.update_layout(
    title="Lunch on math score",
    margin=dict(l=20, r=20, t=70, b=20, pad=1),
    paper_bgcolor="lightgray",
)
fig.show()

# reading 
fig = px.box(df, 
                 x="lunch", y="reading score", 
                 facet_col="gender",
                 color='gender',
                 template='simple_white',
                 width=900, height=400,
                 category_orders={
                     "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]
                     }                
                )

fig.update_layout(
    title="Lunch on reading score",
    margin=dict(l=20, r=20, t=70, b=20, pad=1),
    paper_bgcolor="lightgray",
)
fig.show()

# writing
fig = px.box(df, 
                 x="lunch", y="writing score", 
                 facet_col="gender",
                 color='gender',
                 template='simple_white',
                 width=900, height=400,
                 category_orders={
                     "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]
                     }                
                )

fig.update_layout(
    title="Lunch on writing score",
    margin=dict(l=20, r=20, t=70, b=20, pad=1),
    paper_bgcolor="lightgray",
)
fig.show()

# writing, race
fig = px.box(df, 
                 x="lunch", y="math score", 
                 facet_col="race/ethnicity",
                 color='gender',
                 template='simple_white',
                 width=900, height=400,
                 category_orders={
                     "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]
                     }                
                )

fig.update_layout(
    title="Lunch on math score: variation across race group",
    margin=dict(l=20, r=20, t=70, b=20, pad=1),
    paper_bgcolor="lightgray",
)
fig.show()


# # writing, race, parent education
DF = df[(df['race/ethnicity'] == 'group A') & (df['gender'] == 'female')]

fig = px.box(DF, 
                 x="lunch", y="writing score", 
                 color="parental level of education",
#                  facet_col=
#                  color='gender',
                 template='simple_white',
                 width=900, height=400,
                 category_orders={
                     "race/ethnicity": ["group A", "group B", "group C", "group D", "group E"]
                     }                
                )

fig.update_layout(
    title="Lunch on writing score: variation across ParentEduc. (female students of group A) ",
    margin=dict(l=20, r=20, t=70, b=20, pad=1),
    paper_bgcolor="lightgray",
)
fig.show()

<a id="6"></a>
<font color="lightseagreen" size=+2.5><b>6. Test score correlation?</b></font>

<font color="lightseagreen" size=+1.5><b>Are test score correlated/related with one another?</b></font>

- Reading and writing test scores seem to be correlated with each other for both gender groups
- However, the correlation of maths test scores with reading and writing tests is different for the two gender groups. For female students their reading & writing scores bettered their math score while the opposite is true for male students.


<a href="#toc">Back to top</a>

In [ ]:
templates  = ["plotly", "plotly_white", "plotly_dark", "ggplot2", "seaborn", "simple_white", "none"]

fig = px.density_contour(df, x="math score", y="reading score", color="gender",trendline="ols",
                 marginal_x="histogram", marginal_y="histogram",
                 #hover_data=['race/ethnicity'],
                         width=600,height=600,
                 title= '<b> Math vs Reading <b>',
                 template="simple_white",
                        color_discrete_map={ 
                    "male": "RebeccaPurple", "female": "lightsalmon"
                },
                        )
fig.layout.update(titlefont={'color': 'black', 'size': 24},
                  paper_bgcolor='#ececec',
                  plot_bgcolor='#ececec',
                 )
fig.show()

#######


fig =px.density_contour(df, x="math score", y="writing score", color="gender",trendline="ols",
                 marginal_x="histogram", marginal_y="histogram",
                 #hover_data=['race/ethnicity'],
                        width=600,height=600,
                 title= '<b> Math vs Writing <b>',
                 template="simple_white",
                       color_discrete_map={ 
                    "male": "RebeccaPurple", "female": "lightsalmon"
                },
                       )
fig.layout.update(titlefont={'color': 'black', 'size': 24},
                  paper_bgcolor='#ececec',
                  plot_bgcolor='#ececec',
                 )
fig.show()

######
fig = px.density_contour(df, x="reading score", y="writing score", color="gender",trendline="ols",
                 marginal_x="histogram", marginal_y="histogram",
                 #hover_data=['race/ethnicity'],
                         width=600,height=600,
                 title= '<b> Reading vs Writing <b>',
                 template="simple_white", 
                               color_discrete_map={ 
                    "male": "RebeccaPurple", "female": "lightsalmon"
                },
                        )
fig.layout.update(titlefont={'color': 'black', 'size': 24},
                  paper_bgcolor='#ececec',
                  plot_bgcolor='#ececec',
                 )
fig.show()


<a id="7"></a>
<font color="lightseagreen" size=+2.5><b>7. Correlation heatmap</b></font>

* Here we use the point-biserial correlation coefficient to check the correlation between the independent variables (categoricals) vs continuous targets. 
* We notice that the correlations are at best weak. The highest correlation is *lunch* vs *math test score* at 0.38.


In [ ]:
data= data.copy()

num_cols =['math score', 'reading score', 'writing score', 'average_score']
cat_cols = ['gender', 'race/ethnicity', 'lunch', 'parental level of education', 'test preparation course']
feats = ['gender', 'lunch', 'test preparation course', 'math score', 'writing score', 'reading score', 'race/ethnicity', 'parental level of education']

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

le_data = data.copy()

for col in feats:
    le_data[col] = le.fit_transform(data[col])
    
train = le_data

In [ ]:
def point_biserial(x, y):
    pb = stats.pointbiserialr(x, y)
    return pb[0]

rows= []
for x in feats[:-2]:
    col = []
    for y in feats[:-2] :
        pbs =point_biserial(train[x], train[y]) 
        col.append(round(pbs,2))  
    rows.append(col)  

    
pbs_results = np.array(rows)
DF = pd.DataFrame(pbs_results, columns = train[feats[:-2]].columns, index =train[feats[:-2]].columns)

mask = np.triu(np.ones_like(DF, dtype=bool))
DF=DF.mask(mask)

fig = go.Figure(data= go.Heatmap(z=DF,
                  x=DF.index.values,
                  y=DF.columns.values,       
                  xgap=3, ygap=3,
                  colorscale='greens',
                  colorbar_thickness=10,
                  colorbar_ticklen=3,
                   )
                )
fig.update_layout(title_text="Correlation heatmap", 
                title_x=0.5,
                font_family="San Serif",
                titlefont={'size': 24},
                width=600, height=500,
                xaxis_showgrid=False,
                yaxis_showgrid=False,
                yaxis_autorange='reversed', 
                paper_bgcolor='#ececec',
                margin=dict(l=70, r=70, t=70, b=70, pad=1),
                template="simple_white"    )

fig.add_vrect(
    x0=-0.5, x1=2.5, y0=0, y1=0.5,
    fillcolor='red', opacity=0.95,
    layer="below", line_width=0,
)



fig.show()

<a id="8"></a>
<font color="lightseagreen" size=+2.5><b>8. Conclusions: EDA</b></font>

- Both groups of students have their strengths and weaknesses in test performances. Male students were better in maths and females in the other two test subjects.
- Tutorials/test preparation courses helped improve their score. It helped male students in their writing test especially those from race group B and C whose parents have 'some college' education.
- Lunch type also had an effect on students' test performance. A reduced or free lunch had an influence (negative) on the math score of race group A and C female students.
- Race and parental education level also seem to have an effect on the students test score. Generally race group E were the better performers while race group A being the low achievers. Students whose parents have a high level education scored the better scores, generally.

<a href="#toc">Back to top</a>

---

<font color="lightgray" size=+3.0><b>Part-2: Model</b></font>

<a id="9"></a>
<font color="lightseagreen" size=+2.5><b>9. Modeling and prediction</b></font>

In this section we will try to see if we can predict the test scores (for the three test subjects separately) from the categorical input features. We suspect that the model accuracy may not be high just by looking at what kind of features we have and from experience (we all have been students at least once in our lifetime). Let's see.
- Input features : gender, race/ethnicity, lunch, parental level of education, and test preparation course
- Target variable: maths, witing and reading test scores 
- We use a simple random forest regression model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import shap 

<font color="lightseagreen" size=+1.5><b>Outliers treatment: Z score </b></font>

* Since we have near normal distribution we can and will use Z-score method to detect and deal with outliers.
* In a normally distributed data 99.7% of the data lie within +/- 3 standard deviation. 
* So we treat data which are outside of +/- three standard deviations as outliers.

In [ ]:
upper_limit_math = data['math score'].mean() + 3*data['math score'].std()
lower_limit_math = data['math score'].mean() - 3*data['math score'].std()
upper_limit_reading = data['reading score'].mean() + 3*data['reading score'].std()
lower_limit_reading = data['reading score'].mean() - 3*data['reading score'].std()
upper_limit_writing = data['writing score'].mean() + 3*data['writing score'].std()
lower_limit_writing = data['writing score'].mean() - 3*data['writing score'].std()

print('The boundaries for the outliers are as follows:\n')
print("Upper boundary math_score: ",np.round(upper_limit_math, 2))
print("Lower boundary math_score: ",np.round(lower_limit_math, 2))
print('\n')
print("Upper boundary reading_score: ",np.round(upper_limit_reading, 2))
print("Lower boundary reading_score: ",np.round(lower_limit_reading, 2))
print('\n')
print("Upper boundary writing_score: ",np.round(upper_limit_writing, 2))
print("Lower boundary writing_score: ",np.round(lower_limit_writing, 2))

<font color="lightseagreen" size=+1.5><b>Flooring/Capping outliers </b></font>

After identifying possible outlier values, we make a decision to either drop them or cap/floor them to the calculated limits. Here we made the decision of capping/flooring. What we can also try is to do separate analysis with and without the outliers and compare the results to see the effect of dropping them. We will not do it in this notebook however.


In [ ]:
#math
data['math score'] = np.where(
    data['math score'] > upper_limit_math,
    upper_limit_math,
    np.where(
        data['math score'] < lower_limit_math,
        lower_limit_math,
        data['math score']
    )
)
#reading
data['reading score'] = np.where(
    data['reading score'] > upper_limit_reading,
    upper_limit_reading,
    np.where(
        data['reading score'] < lower_limit_reading,
        lower_limit_reading,
        data['reading score']
    )
)
#writing 
data['writing score'] = np.where(
    data['writing score'] > upper_limit_writing,
    upper_limit_writing,
    np.where(
        data['writing score'] < lower_limit_writing,
        lower_limit_writing,
        data['writing score']
    )
)

<font color="lightseagreen" size=+1.5><b> Define X's and y's </b></font>

* The X's are the in put features
* The y's are the individual test scores (math, reading and writing)

In [ ]:
# features/independent variables; X's
X = train[cat_cols]

# the three targets; y's
y_m = train['math score']
y_w = train['writing score']
y_r = train['reading score']

<font color="lightseagreen" size=+1.5><b> Helper functions for model and SHAP </b></font>

Let's make functions for our model prediction and SHAP value calculation.

In [ ]:
def make_model(X, y):
    #train_test_split
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=100)
    # model selection and , we do here randomforest
    rf = RandomForestRegressor(n_estimators=500, max_depth=6, random_state=100)
    # fit model
    rf.fit(X_train, y_train)
    # predict
    predict = rf.predict(X_val)
    # model evaluation metric
    rmse = np.sqrt(mean_squared_error(y_val, predict))
    r_squared = (rf.fit(X_train, y_train).score(X_val, y_val))
    return ('rmse score: {0:.3}'.format(rmse), 'r-squared value is: {0:.3}'.format(r_squared))


In [ ]:
def shap_and_feature_importance(X, y, features):
    # split
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=100)
    # model
    rf = RandomForestRegressor(n_estimators=500, max_depth=6, random_state=100)
    # fit model
    rf.fit(X_train, y_train)    
    # create object that can calculate shap values
    explainer = shap.TreeExplainer(rf)
    # calculate Shap values
    shap_values = explainer.shap_values(X_train)
    # feature importance plot
    shap.summary_plot(shap_values, X_train, feature_names=features, plot_type="bar")
    # shap summary plot
    shap.summary_plot(shap_values, X_train, feature_names=features)    
    return 

<font color="lightseagreen" size=+1.5><b> Model accuracy </b></font>

Since we are dealing with a regression problem, we will consider `RMSE and r-squared` values as metrics to evaluate our model. We will note that both scores are not very good, ie, high rmse value (`very close to the standard deviations of the test scores`) and too low r-squared scores. 

In [ ]:
scores = [y_m, y_w, y_r]
subjects =['math', 'writing', 'reading']

for sub, item in zip(subjects, scores):
    print(sub + ' score =>', make_model(X, item))
    print('standard deviation of ' + sub , np.round(data[sub + ' score'].std(), 2))

<font color="lightseagreen" size=+1.5><b>Feature importance and SHAP summary plots</b></font>

* From the feature importance plots we see that `lunch (math, reading)`, `test prep (writing)`, `gender (2nd reading & writing)` and `race (2nd math)` are important features for the model. However, the model does not see the importance of parental education in predicting students' scores. Even though it seems counterintuitive, it actually makes sense (for me). It doesn't really matter how many degrees parents have. What is important is how much time they can dedicate to their kids in guiding them.


In [ ]:
scores = [y_m, y_w, y_r]
subjects =['maths', 'writing', 'reading']

for sub, item in zip(subjects, scores):
    print(' ')
    print('Feature importance and SHAP summary for ' + sub + ' test score')
    print(shap_and_feature_importance(X, item, features=cat_cols))   

<a id="10"></a>
<font color="lightseagreen" size=+2.5><b>10. Conclusions: Model</b></font>

The model accuracy (judged by the `r-squared and RMSE values`) is not very high. But this was kind of expected as the features we have are not highly predictive of school/test performance. Test performance may depend on more features than given in this dataset. Therefore, more features are required to accurately identify what affects students' test performance. Among other, the following features could be thought of as additional features that could have an impact on students test score. 

- study hour per week (hrs)
- interest for the subject (low, medium, high)
- get help from parents (yes or no)
- is first child ? (yes or no)
- number of siblings
- marital status of parents (married, divorced, widowed)
- practice sport? yes or no
- and possibly more

So the model accuracy and prediction should be read/taken with cautions! After all, the dataset is synthetic.


<a id="11"></a>
<font color="lightseagreen" size=+2.5><b>11. Closing remarks</b></font>

<font color="lightseagreen" size=+1.5><b>We would like to close the loop by answering the three question we asked at the start.</b></font>

* How effective is the test preparation course?
> Test preparation helps indeed. `Male` students increased their `writing score` markedly by having the test preparation course.
* Which major factors contribute to test outcomes?
> `Lunch`, `test preparation` and `gender`. Male students better their female colleagues on math tests. The female students did so on the writing/reading tests. Lunch had the highest effect on math tests.
* What would be the best way to improve student scores on each test?
> See above. However, as discussed in the model conclusion part, the `given features are not sufficient` to pinpoint what factors affect students' test performance.
 

<a id="12"></a>
<font color="lightseagreen" size=+2.5><b>12. References</b></font>

1. [Plotly official webpage](https://plotly.com/)

2. [Royce Kimmons' (the dataset creator) webpage](http://roycekimmons.com/tools/generated_data/exams)

3. [Sckit-learn official webpage](https://scikit-learn.org/stable/index.html)

4. [An introduction to explainable AI with Shapley values](https://shap.readthedocs.io/en/latest/example_notebooks/overviews/An%20introduction%20to%20explainable%20AI%20with%20Shapley%20values.html)


<font color="lightseagreen" size=+1.5><b>Thank you for reading this notebook!</b></font>

<font color="lightseagreen" size=+1.5><b>If you have remarks and/or questions I am eager to hear it; please drop them in the comments below.</b></font>

<a href="#toc">Back to top</a>
